**NGC 752**

In [ ]:
# update for NGC_752: #MIST isochrones
# install if needed
!pip install -q isochrones
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import types
from isochrones import get_ichrone
#load data
STAR_FILE = "final!_errors.csv"
df = pd.read_csv(STAR_FILE, low_memory=False)
# cluster name
cluster_col = next((c for c in df.columns if "cluster" in c.lower()), None)
if cluster_col is None:
    raise ValueError("Could not find a cluster column in final!_errors.csv")
mask_ngc752 = df[cluster_col].astype(str).str.contains("NGC_752", case=False, na=False)
df_ngc752 = df.loc[mask_ngc752].copy()
print(f"NGC 752 stars found: {len(df_ngc752)}")
# helper to pick columns
def pick(col_patterns):
    cols = df_ngc752.columns
    cols_lower = cols.str.lower()
    for col, cl in zip(cols, cols_lower):
        if all(p.lower() in cl for p in col_patterns):
            return col
    return None
# gaia mags
g_col = pick(["gmag"]) or pick(["phot_g"]) or pick(["g_mean_mag"])
bp_col = pick(["bp", "mag"])
rp_col = pick(["rp", "mag"])
if g_col is None or bp_col is None or rp_col is None:
    raise ValueError(f"Could not find G/BP/RP columns: g={g_col}, bp={bp_col}, rp={rp_col}")
# astrometry columns
plx_col = pick(["plx"]) or pick(["parallax"])
pmra_col = pick(["pmra"])
pmdec_col = pick(["pmde"]) or pick(["pmdec"])
if plx_col is None or pmra_col is None or pmdec_col is None:
    raise ValueError(f"Could not find astrometry columns: plx={plx_col}, pmra={pmra_col}, pmdec={pmdec_col}")
# colors
df_ngc752["BP_RP"] = df_ngc752[bp_col] - df_ngc752[rp_col]
color_cl = df_ngc752["BP_RP"].to_numpy()
g_mag_cl = pd.to_numeric(df_ngc752[g_col], errors="coerce").to_numpy()
good = np.isfinite(color_cl) & np.isfinite(g_mag_cl)
print(f"→ Plotted {good.sum()} stars from NGC 752")
# define x stars
def robust_stats(series):
    s = pd.to_numeric(series, errors="coerce")
    med = np.nanmedian(s)
    mad = np.nanmedian(np.abs(s - med))
    sigma = 1.4826 * mad if mad > 0 else np.nan
    return med, sigma
# cluster stats
med_plx, sig_plx = robust_stats(df_ngc752[plx_col])
med_pmra, sig_pmra = robust_stats(df_ngc752[pmra_col])
med_pmde, sig_pmde = robust_stats(df_ngc752[pmdec_col])
df_ngc752["dPlx"] = df_ngc752[plx_col] - med_plx
df_ngc752["dpmRA"] = df_ngc752[pmra_col] - med_pmra
df_ngc752["dpmDE"] = df_ngc752[pmdec_col]- med_pmde
df_ngc752["zPlx"] = df_ngc752["dPlx"] / sig_plx if np.isfinite(sig_plx) and sig_plx > 0 else np.nan
df_ngc752["zpmRA"] = df_ngc752["dpmRA"] / sig_pmra if np.isfinite(sig_pmra) and sig_pmra > 0 else np.nan
df_ngc752["zpmDE"] = df_ngc752["dpmDE"] / sig_pmde if np.isfinite(sig_pmde) and sig_pmde > 0 else np.nan
z_stack = np.vstack([
    np.abs(df_ngc752["zPlx"].to_numpy()),
    np.abs(df_ngc752["zpmRA"].to_numpy()),
    np.abs(df_ngc752["zpmDE"].to_numpy())
])
max_abs_z = np.nanmax(z_stack, axis=0)
# "Bad" definition: 10 <= G <= 11 AND max|z| >= 4
slice_10_11 = (g_mag_cl >= 10.0) & (g_mag_cl <= 11.0)
bad_mask = slice_10_11 & np.isfinite(max_abs_z) & (max_abs_z >= 4.0)
print(f"Stars with {g_col} between 10 and 11: {slice_10_11.sum()}")
print(f' → "Bad" (max|z| ≥ 4): {bad_mask.sum()}')
# MIST isochrones
mist = get_ichrone('mist', bands=['G', 'BP', 'RP'])
def patched_interp_value(self, pars, props):
    i0, i1, i2, i3, i4 = self.param_index_order
    pars = [pars[i0], pars[i1], pars[i2]]
    return self.model_grid.interp(pars, "all")
mist.interp_value = types.MethodType(patched_interp_value, mist)
# Gaia DR3-like extinction coefficients
# NGC 752 global params

K_G  = 2.74   
K_BP = 3.37  
K_RP = 2.04  

EBV_GLOBAL = 0.035   
DM_GLOBAL  = 8.30    

ages_gyr = [1.3, 1.4, 1.5, 1.6] 
feh      = 0.1               

plt.figure(figsize=(9, 8))
# Plot all targets
plt.scatter(color_cl[good], g_mag_cl[good], s=15, color='gray',
            edgecolors='k', linewidth=0.5, alpha=0.8,
            label='NGC 752')
# overlay red X on bad ones
bad_good = good & bad_mask
if bad_good.any():
    plt.scatter(color_cl[bad_good], g_mag_cl[bad_good],
                marker='x', s=80, linewidth=2,
                color='red', label='Likely non-members')
# Isochrones
colors = plt.cm.viridis(np.linspace(0, 1, len(ages_gyr)))
for i, age_gyr in enumerate(ages_gyr):
    logage = np.log10(age_gyr * 1e9)
    iso = mist.isochrone(logage, feh)
    col_fit = (iso['BP_mag'] - iso['RP_mag']) + (K_BP - K_RP) * EBV_GLOBAL
    mag_fit = iso['G_mag'] + DM_GLOBAL + K_G * EBV_GLOBAL
    label = f"{age_gyr:.2f} Gyr"
    plt.plot(col_fit, mag_fit, color=colors[i], lw=2.5, label=label)
plt.gca().invert_yaxis()
plt.xlim(0.0, 1.5)
plt.ylim(14,8)
plt.xlabel(r'$G_{\rm BP} - G_{\rm RP}$ (mag)')
plt.ylabel(r'$G$ (mag)')
plt.legend(fontsize=9, loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**THEIA 6046**

In [ ]:
# update for theia_6046: #MIST isochrones
# install if needed
!pip install -q isochrones
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import types
from isochrones import get_ichrone

# load data
STAR_FILE = "final!_errors.csv"
df = pd.read_csv(STAR_FILE, low_memory=False)

# cluster name
cluster_col = next((c for c in df.columns if "cluster" in c.lower()), None)
if cluster_col is None:
    raise ValueError("Could not find a cluster column in final!_errors.csv")

mask_theia6046 = df[cluster_col].astype(str).str.contains("theia_6046", case=False, na=False)
df_theia6046 = df.loc[mask_theia6046].copy()
print(f"theia_6046 stars found: {len(df_theia6046)}")

# helper to pick columns
def pick(col_patterns):
    cols = df_theia6046.columns
    cols_lower = cols.str.lower()
    for col, cl in zip(cols, cols_lower):
        if all(p.lower() in cl for p in col_patterns):
            return col
    return None

# gaia mags
g_col  = pick(["gmag"]) or pick(["phot_g"]) or pick(["g_mean_mag"])
bp_col = pick(["bp", "mag"])
rp_col = pick(["rp", "mag"])
if g_col is None or bp_col is None or rp_col is None:
    raise ValueError(f"Could not find G/BP/RP columns: g={g_col}, bp={bp_col}, rp={rp_col}")

# astrometry columns
plx_col  = pick(["plx"])  or pick(["parallax"])
pmra_col = pick(["pmra"])
pmdec_col = pick(["pmde"]) or pick(["pmdec"])
if plx_col is None or pmra_col is None or pmdec_col is None:
    raise ValueError(f"Could not find astrometry columns: plx={plx_col}, pmra={pmra_col}, pmdec={pmdec_col}")

# colors
df_theia6046["BP_RP"] = df_theia6046[bp_col] - df_theia6046[rp_col]
color_cl = df_theia6046["BP_RP"].to_numpy()
g_mag_cl = pd.to_numeric(df_theia6046[g_col], errors="coerce").to_numpy()
good = np.isfinite(color_cl) & np.isfinite(g_mag_cl)
print(f"→ Plotted {good.sum()} stars from theia_6046")

# define x stars
def robust_stats(series):
    s = pd.to_numeric(series, errors="coerce")
    med = np.nanmedian(s)
    mad = np.nanmedian(np.abs(s - med))
    sigma = 1.4826 * mad if mad > 0 else np.nan
    return med, sigma

# cluster stats
med_plx,  sig_plx  = robust_stats(df_theia6046[plx_col])
med_pmra, sig_pmra = robust_stats(df_theia6046[pmra_col])
med_pmde, sig_pmde = robust_stats(df_theia6046[pmdec_col])

df_theia6046["dPlx"]  = df_theia6046[plx_col]  - med_plx
df_theia6046["dpmRA"] = df_theia6046[pmra_col] - med_pmra
df_theia6046["dpmDE"] = df_theia6046[pmdec_col]- med_pmde

df_theia6046["zPlx"]  = df_theia6046["dPlx"]  / sig_plx  if np.isfinite(sig_plx)  and sig_plx  > 0 else np.nan
df_theia6046["zpmRA"] = df_theia6046["dpmRA"] / sig_pmra if np.isfinite(sig_pmra) and sig_pmra > 0 else np.nan
df_theia6046["zpmDE"] = df_theia6046["dpmDE"] / sig_pmde if np.isfinite(sig_pmde) and sig_pmde > 0 else np.nan

z_stack = np.vstack([
    np.abs(df_theia6046["zPlx"].to_numpy()),
    np.abs(df_theia6046["zpmRA"].to_numpy()),
    np.abs(df_theia6046["zpmDE"].to_numpy())
])
max_abs_z = np.nanmax(z_stack, axis=0)

# "Bad" definition: 10 <= G <= 11 AND max|z| >= 2
slice_10_11 = (g_mag_cl >= 10.0) & (g_mag_cl <= 11.0)
bad_mask = slice_10_11 & np.isfinite(max_abs_z) & (max_abs_z >= 2.0)
print(f"Stars with {g_col} between 10 and 11: {slice_10_11.sum()}")
print(f' → "Bad" (max|z| ≥ 2): {bad_mask.sum()}')

# MIST isochrones
mist = get_ichrone('mist', bands=['G', 'BP', 'RP'])

def patched_interp_value(self, pars, props):
    i0, i1, i2, i3, i4 = self.param_index_order
    pars = [pars[i0], pars[i1], pars[i2]]
    return self.model_grid.interp(pars, "all")
mist.interp_value = types.MethodType(patched_interp_value, mist)

# theia_6046 global params
K_G  = 2.74   # A_G   / E(B-V)
K_BP = 3.37   # A_BP  / E(B-V)
K_RP = 2.04   # A_RP  / E(B-V)

EBV_GLOBAL = 0.3
DM_GLOBAL  = 9.9

ages_gyr = [0.3, 0.5, 0.7, 1, 1.5, 3.7]
feh      = -0.06

# text you want for each age
label_text = {
    0.3: "",
    0.5: "~ Seismic Range",
    0.7: "~ Seismic Range",
    1.0: "",
    1.5: "",
    3.7: "Literature Age",
}

# place each label at a chosen fraction along the isochrone
# (0 = start of array, 1 = end). If an age is missing here, it uses 0.25.
label_frac = {
    0.3: 0.30,
    0.5: 0.30,
    0.7: 0.30,
    1.0: 0.30,
    1.5: 0.30,
    3.7: 0.32,
}

# per-age opacity of the isochrone lines (0 = invisible, 1 = solid)
iso_alpha = {
    0.3: 0.20,
    0.5: 1,
    0.7: 1,
    1.0: 0.20,
    1.5: 0.20,
    3.7: 1,
}

plt.figure(figsize=(9, 8))

# Plot all targets
plt.scatter(
    color_cl[good], g_mag_cl[good],
    s=15, color='gray', edgecolors='k', linewidth=0.5, alpha=0.8,
    label='theia_6046'
)

# overlay red X on bad ones
bad_good = good & bad_mask
if bad_good.any():
    plt.scatter(
        color_cl[bad_good], g_mag_cl[bad_good],
        marker='x', s=80, linewidth=2,
        color='red', label='Likely non-members'
    )

# Isochrones with inline labels at a fraction of the curve
colors = plt.cm.viridis(np.linspace(0, 1, len(ages_gyr)))
for i, age_gyr in enumerate(ages_gyr):
    logage = np.log10(age_gyr * 1e9)
    iso = mist.isochrone(logage, feh)
    col_fit = (iso['BP_mag'] - iso['RP_mag']) + (K_BP - K_RP) * EBV_GLOBAL
    mag_fit = iso['G_mag'] + DM_GLOBAL + K_G * EBV_GLOBAL

    # look up opacity for this age; default to 1.0 if missing
    this_alpha = iso_alpha.get(age_gyr, 1.0)

    # plot isochrone
    legend_label = f"{age_gyr:.2f} Gyr"
    plt.plot(
        col_fit,
        mag_fit,
        color=colors[i],
        lw=2.5,
        alpha=this_alpha,
        label=legend_label
    )

    # inline label if text provided for this age
    if age_gyr in label_text:
        frac = label_frac.get(age_gyr, 0.25)  # default 25% along curve
        frac = max(0.0, min(1.0, frac))      

        idx = int(frac * (len(col_fit) - 1))
        x_lab = col_fit[idx]
        y_lab = mag_fit[idx]

        plt.text(
            x_lab + 0.02,   # small x offset
            y_lab - 0.10,   # small y offset 
            label_text[age_gyr],
            fontsize=8,
            color=colors[i],
            alpha=this_alpha  # label matches line opacity
        )

plt.gca().invert_yaxis()
plt.xlim(0.2, 2.5) 
plt.ylim(17, 9)   
ax = plt.gca()
ax.set_aspect('auto')  

plt.xlabel(r'$G_{\rm BP} - G_{\rm RP}$ (mag)')
plt.ylabel(r'$G$ (mag)')
plt.legend(fontsize=9, loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**CASADO ALESSI 1**

In [ ]:
# MIST isochrones
# install if needed 
!pip install -q isochrones  

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import types
from isochrones import get_ichrone

# load data
STAR_FILE = "final!_errors.csv"
df = pd.read_csv(STAR_FILE, low_memory=False)

# cluster name
cluster_col = next((c for c in df.columns if "cluster" in c.lower()), None)
if cluster_col is None:
    raise ValueError("Could not find a cluster column in final!_errors.csv")

mask_ca1 = df[cluster_col].astype(str).str.contains("Casado-Alessi_1", case=False, na=False)
df_ca1 = df.loc[mask_ca1].copy()
print(f"Casado-Alessi 1 stars found: {len(df_ca1)}")

# helper to pick columns 
def pick(col_patterns):
    cols = df_ca1.columns
    cols_lower = cols.str.lower()
    for col, cl in zip(cols, cols_lower):
        if all(p.lower() in cl for p in col_patterns):
            return col
    return None

# gaia mags
g_col  = pick(["gmag"]) or pick(["phot_g"]) or pick(["g_mean_mag"])
bp_col = pick(["bp", "mag"])
rp_col = pick(["rp", "mag"])
if g_col is None or bp_col is None or rp_col is None:
    raise ValueError(f"Could not find G/BP/RP columns: g={g_col}, bp={bp_col}, rp={rp_col}")

# astrometry columns
plx_col   = pick(["plx"]) or pick(["parallax"])
pmra_col  = pick(["pmra"])
pmdec_col = pick(["pmde"]) or pick(["pmdec"])
if plx_col is None or pmra_col is None or pmdec_col is None:
    raise ValueError(f"Could not find astrometry columns: plx={plx_col}, pmra={pmra_col}, pmdec={pmdec_col}")

# colors
df_ca1["BP_RP"] = df_ca1[bp_col] - df_ca1[rp_col]
color_cl = df_ca1["BP_RP"].to_numpy()
g_mag_cl = pd.to_numeric(df_ca1[g_col], errors="coerce").to_numpy()
good = np.isfinite(color_cl) & np.isfinite(g_mag_cl)

print(f"→ Plotted {good.sum()} stars from Casado-Alessi 1")

# define x stars
def robust_stats(series):
    s = pd.to_numeric(series, errors="coerce")
    med = np.nanmedian(s)
    mad = np.nanmedian(np.abs(s - med))
    sigma = 1.4826 * mad if mad > 0 else np.nan
    return med, sigma

# cluster stats
med_plx,  sig_plx  = robust_stats(df_ca1[plx_col])
med_pmra, sig_pmra = robust_stats(df_ca1[pmra_col])
med_pmde, sig_pmde = robust_stats(df_ca1[pmdec_col])

df_ca1["dPlx"]  = df_ca1[plx_col]  - med_plx
df_ca1["dpmRA"] = df_ca1[pmra_col] - med_pmra
df_ca1["dpmDE"] = df_ca1[pmdec_col]- med_pmde

df_ca1["zPlx"]  = df_ca1["dPlx"]  / sig_plx  if np.isfinite(sig_plx)  and sig_plx  > 0 else np.nan
df_ca1["zpmRA"] = df_ca1["dpmRA"] / sig_pmra if np.isfinite(sig_pmra) and sig_pmra > 0 else np.nan
df_ca1["zpmDE"] = df_ca1["dpmDE"] / sig_pmde if np.isfinite(sig_pmde) and sig_pmde > 0 else np.nan

z_stack = np.vstack([
    np.abs(df_ca1["zPlx"].to_numpy()),
    np.abs(df_ca1["zpmRA"].to_numpy()),
    np.abs(df_ca1["zpmDE"].to_numpy())
])
max_abs_z = np.nanmax(z_stack, axis=0)

# "Bad" definition: 10 <= G <= 11 AND max|z| >= 4
slice_10_11 = (g_mag_cl >= 10.0) & (g_mag_cl <= 11.0)
bad_mask = slice_10_11 & np.isfinite(max_abs_z) & (max_abs_z >= 4.0)

print(f"Stars with {g_col} between 10 and 11: {slice_10_11.sum()}")
print(f'  → "Bad" (max|z| ≥ 4): {bad_mask.sum()}')

# MIST isochrones
mist = get_ichrone('mist', bands=['G', 'BP', 'RP'])

def patched_interp_value(self, pars, props):
    i0, i1, i2, i3, i4 = self.param_index_order
    pars = [pars[i0], pars[i1], pars[i2]]
    return self.model_grid.interp(pars, "all")
mist.interp_value = types.MethodType(patched_interp_value, mist)
K_G  = 2.3  
K_BP = 2.9  
K_RP = 2.0  

EBV_GLOBAL = 0.03 
DM_GLOBAL  = 9 

ages_gyr = [0.8, 1.45, 1.7, 2.5, 2.75, 3.0, 3.5]   
feh = 0.02                                      

label_text = {
    0.8:  "Literature Age",
    1.45: "Literature Age",
    1.7:  "~ Seismic Age",
    2.5:  "",
    2.75: "",
    3.0:  "",
    3.5:  "",
}

# position of label along each isochrone (0=start, 1=end)
label_frac = {
    0.8:  0.3,
    1.45: 0.3,
    1.7:  0.3,
    2.5:  0.3,
    2.75: 0.3,
    3.0:  0.3,
    3.5:  0.3,
}

# per-age opacity for lines + labels
iso_alpha = {
    0.8:  0.9,
    1.45: 0.9,
    1.7:  0.9,
    2.5:  0.2,
    2.75: 0.2,
    3.0:  0.2,
    3.5:  0.2,
}

plt.figure(figsize=(9, 8))

# Plot all targets
plt.scatter(
    color_cl[good], g_mag_cl[good],
    s=15, color='gray', edgecolors='k', linewidth=0.5, alpha=0.8,
    label='Casado-Alessi 1'
)

# overlay red X on bad ones
bad_good = good & bad_mask
if bad_good.any():
    plt.scatter(
        color_cl[bad_good], g_mag_cl[bad_good],
        marker='x', s=80, linewidth=2,
        color='red', label='Likely non-members'
    )

# Isochrones
colors = plt.cm.viridis(np.linspace(0, 1, len(ages_gyr)))
for i, age_gyr in enumerate(ages_gyr):
    logage = np.log10(age_gyr * 1e9)
    iso = mist.isochrone(logage, feh)
    
    col_fit = (iso['BP_mag'] - iso['RP_mag']) + (K_BP - K_RP) * EBV_GLOBAL
    mag_fit = iso['G_mag'] + DM_GLOBAL + K_G * EBV_GLOBAL
    
    label = f"{age_gyr:.2f} Gyr"
    this_alpha = iso_alpha.get(age_gyr, 1.0)

    # plot line
    plt.plot(
        col_fit,
        mag_fit,
        color=colors[i],
        lw=2.5,
        alpha=this_alpha,
        label=label
    )

    # inline label if desired
    if age_gyr in label_text:
        frac = label_frac.get(age_gyr, 0.25)
        frac = max(0.0, min(1.0, frac))
        idx = int(frac * (len(col_fit) - 1))
        x_lab = col_fit[idx]
        y_lab = mag_fit[idx]
        plt.text(
            x_lab + 0.02,
            y_lab - 0.10,
            label_text[age_gyr],
            fontsize=8,
            color=colors[i],
            alpha=this_alpha
        )

plt.gca().invert_yaxis()
plt.xlim(0.0, 1.5)
plt.ylim(16,8)

ax = plt.gca()
ax.set_aspect('auto')

plt.xlabel(r'$G_{\rm BP} - G_{\rm RP}$ (mag)')
plt.ylabel(r'$G$ (mag)')
plt.legend(fontsize=9, loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()